# ComfyUI ワークフロー自動実行ノートブック（Google Colab / SDXL ハイブリッド対応）

Google Colab 上で ComfyUI サーバーを起動し、`workflows/` に配置した **API 形式のワークフロー JSON** を使って画像生成を実行・保存するためのノートブックです。  
コア処理は [`comfyui/runner/`](./runner/) にモジュール化されているため、将来的に `colab/illustration-generator` 等の別パイプラインからもそのまま `import` して再利用できます。

### SDXL ハイブリッド運用
- **公開 SDXL ベースモデル / VAE 等**: Hugging Face / Civitai から Colab ローカル SSD へ高速自動 DL（Drive 容量を節約）
- **自作 LoRA / プライベートモデル**: Google Drive（`MyDrive/ComfyUI/models/`）に配置し `extra_model_paths.yaml` でコピーなし参照
- **生成パラメータ群**: `presets/sdxl_default.yaml` 等のプリセットで解像度・Steps・CFG・LoRA 強度・プロンプトを一括制御

### 事前準備
1. Colab のランタイムを **GPU（T4 / L4 / A100 等）** に設定
2. ComfyUI から **API 形式** でワークフロー JSON を書き出し（`Workflow -> Export (API)` または Dev mode 有効化後の `Save (API Format)`）
3. 認証が必要なモデルを DL する場合は、Colab 左メニューの「シークレット（鍵アイコン）」に `HF_TOKEN` / `CIVITAI_API_KEY` を登録

## 1. GPU 確認 & Google Drive マウント

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. パス・設定の定義

Google Drive 上のモデルフォルダやワークフロー配置先を環境に合わせて調整してください。

In [ ]:
from pathlib import Path

# --- リポジトリ設定 ---
REPO_URL = "https://github.com/belre/stable-diffusion-pipelines.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/stable-diffusion-pipelines")

# --- ComfyUI 本体・Drive パス設定 ---
COMFYUI_DIR = Path("/content/ComfyUI")
DRIVE_BASE = Path("/content/drive/MyDrive/ComfyUI")
DRIVE_MODELS_DIR = DRIVE_BASE / "models"        # 自作 LoRA / VAE 等（loras/, checkpoints/, vae/）
DRIVE_WORKFLOWS_DIR = DRIVE_BASE / "workflows"  # Drive 側に JSON を置く場合
DRIVE_INPUT_DIR = DRIVE_BASE / "input"          # LoadImage 用入力画像
DRIVE_OUTPUT_DIR = DRIVE_BASE / "output"        # 生成画像の保存先
DRIVE_MODELS_MANIFEST = DRIVE_BASE / "models.yaml"  # Drive に models.yaml を置く場合

# --- 追加インストールするカスタムノード（必要なものだけコメント解除・追記） ---
CUSTOM_NODES = [
    # "https://github.com/ltdrdata/ComfyUI-Manager.git",
]

# --- ComfyUI 起動オプション ---
COMFYUI_HOST = "127.0.0.1"
COMFYUI_PORT = 8188
COMFYUI_EXTRA_ARGS = []  # T4(15GB) で SDXL を使う場合は通常空で OK。VRAM が厳しい場合は ["--lowvram"]

print("REPO_DIR         :", REPO_DIR)
print("COMFYUI_DIR      :", COMFYUI_DIR)
print("DRIVE_MODELS_DIR :", DRIVE_MODELS_DIR)

## 3. リポジトリ取得 & `runner` パッケージ準備

In [ ]:
import sys

if not REPO_DIR.exists():
    !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

RUNNER_ROOT = REPO_DIR / "comfyui"
!pip install -q -r {RUNNER_ROOT / 'requirements.txt'}

if str(RUNNER_ROOT) not in sys.path:
    sys.path.insert(0, str(RUNNER_ROOT))

from runner import (
    ComfyUIClient,
    ComfyUIServer,
    ModelSpec,
    check_workflow_models,
    download_models,
    generate_extra_model_paths_yaml,
    load_model_manifest,
    load_workflow,
)
print("runner モジュールの読み込みに成功しました。")

## 4. ComfyUI 本体のセットアップ & SDXL ハイブリッドモデル準備

1. ComfyUI 本体を `/content/ComfyUI` にインストール
2. 高速 DL 用の `aria2` を導入し、公開 SDXL モデルを `/content/ComfyUI/models/` へ自動ダウンロード（既にローカルまたは Drive に存在する場合はスキップ）
3. `extra_model_paths.yaml` を生成し、Google Drive 上の自作 LoRA 等をコピーなしで認識させる

In [ ]:
if not COMFYUI_DIR.exists():
    !git clone https://github.com/comfyanonymous/ComfyUI.git {COMFYUI_DIR}
    !pip install -q -r {COMFYUI_DIR / 'requirements.txt'}
else:
    print("ComfyUI は既にインストール済みです:", COMFYUI_DIR)

# カスタムノードの導入（指定がある場合）
custom_nodes_dir = COMFYUI_DIR / "custom_nodes"
for repo_url in CUSTOM_NODES:
    node_name = repo_url.rstrip("/").split("/")[-1].removesuffix(".git")
    target_dir = custom_nodes_dir / node_name
    if not target_dir.exists():
        !git clone {repo_url} {target_dir}
        req = target_dir / "requirements.txt"
        if req.is_file():
            !pip install -q -r {req}

# Google Drive のモデルフォルダを ComfyUI に認識させる（自作 LoRA 等向け）
EXTRA_MODEL_PATHS = None
if DRIVE_MODELS_DIR.is_dir():
    EXTRA_MODEL_PATHS = generate_extra_model_paths_yaml(
        drive_models_dir=DRIVE_MODELS_DIR,
        output_path=COMFYUI_DIR / "extra_model_paths_colab.yaml",
    )
    print("extra_model_paths を生成しました:", EXTRA_MODEL_PATHS)
else:
    print(f"[INFO] {DRIVE_MODELS_DIR} が見つかりません。自作 LoRA 等を使う場合は Drive に作成してください。")

In [ ]:
# --- 公開 SDXL モデルの自動ダウンロード（Hugging Face / Civitai） ---
!apt-get -qq update && apt-get -qq install -y aria2

# Colab Secrets からトークンを取得（未設定でもエラーにならない）
def get_secret(name: str) -> str | None:
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

HF_TOKEN = get_secret("HF_TOKEN")
CIVITAI_API_KEY = get_secret("CIVITAI_API_KEY")
print("HF_TOKEN        :", "設定あり" if HF_TOKEN else "未設定")
print("CIVITAI_API_KEY :", "設定あり" if CIVITAI_API_KEY else "未設定")

# 1) Drive に models.yaml がある場合はそこから読み込む
model_specs: list[ModelSpec] = []
if DRIVE_MODELS_MANIFEST.is_file():
    model_specs.extend(load_model_manifest(DRIVE_MODELS_MANIFEST))
    print(f"{DRIVE_MODELS_MANIFEST} から {len(model_specs)} 件のモデル定義を読み込みました。")

# 2) ノートブック上で直接指定したいモデルがあればここに追記（必要なものだけコメント解除）
INLINE_MODELS: list[ModelSpec] = [
    # 例: Animagine XL 3.1 (Hugging Face)
    # ModelSpec(
    #     subdir="checkpoints",
    #     filename="animagine-xl-3.1.safetensors",
    #     hf_repo="cagliostrolab/animagine-xl-3.1",
    #     hf_file="animagine-xl-3.1.safetensors",
    # ),
    # 例: Civitai モデル
    # ModelSpec(
    #     subdir="checkpoints",
    #     filename="my_sdxl_model.safetensors",
    #     url="https://civitai.com/api/download/models/000000",
    # ),
]
model_specs.extend(INLINE_MODELS)

if model_specs:
    download_models(
        model_specs,
        comfyui_dir=COMFYUI_DIR,
        hf_token=HF_TOKEN,
        civitai_token=CIVITAI_API_KEY,
        drive_models_dir=DRIVE_MODELS_DIR if DRIVE_MODELS_DIR.is_dir() else None,
    )
else:
    print("自動 DL 対象のモデルは未指定です（Drive 側モデルのみを使用します）。")

## 5. ComfyUI サーバー起動（バックグラウンド）

In [ ]:
server = ComfyUIServer(
    comfyui_dir=COMFYUI_DIR,
    host=COMFYUI_HOST,
    port=COMFYUI_PORT,
    extra_args=COMFYUI_EXTRA_ARGS,
    extra_model_paths_config=EXTRA_MODEL_PATHS,
)
client = server.start(wait_ready=True, timeout=180.0)
print("ComfyUI サーバー起動完了:", server.base_url)

## 6. SDXL プリセット選択 & ワークフロー JSON の実行

`WORKFLOWS_SOURCE_DIR` に置いた API 形式 JSON を順番に実行します。  
- `PRESET_PATH`: `presets/sdxl_default.yaml`（縦長 832x1216）や `presets/sdxl_square.yaml`（1024x1024）を指定すると SDXL パラメータを一括適用できます（`None` なら JSON 内の値をそのまま使用）。
- `CLIPTextEncode` と `CLIPTextEncodeSDXL`（`text_g` / `text_l`）の両方に対応しています。

In [ ]:
# リポジトリ内の comfyui/workflows を使うか、Drive の workflows を使うか選択
WORKFLOWS_SOURCE_DIR = DRIVE_WORKFLOWS_DIR if DRIVE_WORKFLOWS_DIR.is_dir() else (RUNNER_ROOT / "workflows")
INPUT_DIR = DRIVE_INPUT_DIR if DRIVE_INPUT_DIR.is_dir() else (RUNNER_ROOT / "input")
OUTPUT_DIR = Path("/content/comfyui_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- SDXL プリセット & パラメータ上書き設定 ---
PRESET_PATH = RUNNER_ROOT / "presets" / "sdxl_default.yaml"  # 使わない場合は None
POSITIVE_PROMPT = None  # 例: "1girl, solo, anime style, masterpiece, best quality"
NEGATIVE_PROMPT = None  # None の場合はプリセットまたは JSON の値を使用
RANDOMIZE_SEED = True   # True にすると実行ごとにシード値をランダム化
FIXED_SEED = None       # 固定シードを使いたい場合は整数を指定（RANDOMIZE_SEED より優先）
REPEAT = 1              # 各ワークフローの実行回数
OVERRIDES = {
    # 例: "class:CheckpointLoaderSimple.ckpt_name": "animagine-xl-3.1.safetensors",
    # 例: "class:LoraLoader.lora_name": "my_lora.safetensors",
    # 例: "class:KSampler.steps": 28,
}

# --- 事前チェック: ワークフローが参照するモデルがローカル/Drive に揃っているか確認 ---
for wf_file in sorted(WORKFLOWS_SOURCE_DIR.glob("*.json")):
    check = check_workflow_models(
        load_workflow(wf_file),
        comfyui_dir=COMFYUI_DIR,
        drive_models_dir=DRIVE_MODELS_DIR if DRIVE_MODELS_DIR.is_dir() else None,
    )
    if check["missing"]:
        print(f"[WARN] {wf_file.name} で未配置のモデルがあります: {check['missing']}")
    else:
        print(f"[OK] {wf_file.name}: 参照モデル確認済み ({len(check['found'])} 件)")

def on_progress(value: int, max_value: int, node: str | None) -> None:
    print(f"\r  [step {value}/{max_value}] node={node}", end="", flush=True)

results = client.run_directory(
    workflows_dir=WORKFLOWS_SOURCE_DIR,
    output_dir=OUTPUT_DIR,
    preset=PRESET_PATH,
    overrides=OVERRIDES or None,
    randomize_seed=RANDOMIZE_SEED,
    seed=FIXED_SEED,
    positive_prompt=POSITIVE_PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    input_dir=INPUT_DIR if INPUT_DIR.is_dir() else None,
    repeat=REPEAT,
    progress_callback=on_progress,
)
print()
print(f"実行完了: {len(results)} ジョブ")
for r in results:
    print(f"- {r.workflow_path.name if r.workflow_path else r.prompt_id}: {len(r.output_files)} file(s), {r.elapsed_seconds:.1f}s")

## 7. 生成画像のプレビュー & Google Drive への保存

In [ ]:
from IPython.display import display
from PIL import Image

generated_files = [f for r in results for f in r.output_files]
if not generated_files:
    print("生成された画像がありません。workflows フォルダに API 形式 JSON があるか確認してください。")
else:
    for img_path in generated_files:
        print(img_path.name)
        if img_path.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}:
            display(Image.open(img_path))

In [ ]:
import shutil

if generated_files:
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for img_path in generated_files:
        dst = DRIVE_OUTPUT_DIR / img_path.name
        shutil.copy2(img_path, dst)
        print("Drive に保存:", dst)

## 8. サーバー停止（作業終了時）

In [ ]:
server.stop()
print("ComfyUI サーバーを停止しました。")